In [1]:
# Notebook 02.5: Stage 1 Hyperparameter Search (per backbone), FINAL RUN
# Fixes vs previous run:
#  - per-architecture seed (was: one shared seed -> identical candidate menus)
#  - cudnn.benchmark disabled (was: non-deterministic conv algo selection)
#  - 20 trials, more of them in TPE-guided phase not random startup
#  - proxy budget raised to 8 epochs / patience 3

import os, json, time, gc
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import timm

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"optuna  : {optuna.__version__}")

import sys
sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
from abmil_common import (
    build_backbone, PatchDataset, PatchClassifier, get_normalisation_tensors
)

PyTorch : 2.10.0+cu128
CUDA    : True
optuna  : 4.9.0


In [2]:
# Paths (NB02 only)
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
OUT  = Path("/kaggle/working")

X_TRAIN_PATH       = NB02 / "X_train_patches.npy"
Y_TRAIN_PATH       = NB02 / "y_train_labels.npy"
BAG_IDS_TRAIN_PATH = NB02 / "bag_ids_train.npy"
FOLD_IDS_PATH      = NB02 / "fold_ids.npy"
CLASS_WEIGHTS_PATH = NB02 / "class_weights.json"

In [3]:
# Config
BASE_SEED    = 42
STAGE1_FOLD  = 0
PATCH_SIZE   = 224
BS_STAGE1    = 128

N_TRIALS_STAGE1 = 20   # same budget for all three (fairness, not a per-architecture judgment call)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CHANGED: disabled (this was a real source of run-to-run non-determinism)
torch.backends.cudnn.benchmark = False

print("Device:", DEVICE)
print("CPU cores available:", os.cpu_count())

_mean_gpu, _std_gpu = get_normalisation_tensors(DEVICE)

Device: cuda
CPU cores available: 4


In [4]:
# Load NB02 outputs and rebuild the inner-validation split
# (fold 0 excluded entirely, fold 1 used as inner validation, unchanged from before)
X_train_all  = np.load(X_TRAIN_PATH)
y_train_all  = np.load(Y_TRAIN_PATH)
bag_ids_all  = np.load(BAG_IDS_TRAIN_PATH)
fold_ids     = np.load(FOLD_IDS_PATH)

with open(CLASS_WEIGHTS_PATH) as f:
    raw_cw = json.load(f)
class_weight_dict = {int(k): float(v) for k, v in raw_cw.items()}

INNER_VAL_FOLD = 1

all_bags = np.unique(bag_ids_all)
search_bags = all_bags[fold_ids[all_bags] != STAGE1_FOLD]

train_bag_indices = search_bags[fold_ids[search_bags] != INNER_VAL_FOLD]
val_bag_indices   = search_bags[fold_ids[search_bags] == INNER_VAL_FOLD]

train_mask = np.isin(bag_ids_all, train_bag_indices)
val_mask   = np.isin(bag_ids_all, val_bag_indices)

X_tr, y_tr = X_train_all[train_mask], y_train_all[train_mask]
X_val, y_val = X_train_all[val_mask], y_train_all[val_mask]

print(f"Search train bags (folds != 0, != {INNER_VAL_FOLD}): {len(train_bag_indices)}")
print(f"Search val bags   (fold {INNER_VAL_FOLD}): {len(val_bag_indices)}")
print(f"Train patches: {X_tr.shape}  Val patches: {X_val.shape}")

Search train bags (folds != 0, != 1): 746
Search val bags   (fold 1): 239
Train patches: (35787, 224, 224, 1)  Val patches: (11465, 224, 224, 1)


In [5]:
# Stage 1 Optuna objective (CHANGED: max_epochs 6->8, patience 2->3)
def make_stage1_objective(model_name, X_tr_, y_tr_, X_val_, y_val_,
                           class_weight_dict_, seed, max_epochs=8, patience=3):

    def objective(trial):
        # CHANGED: per-trial seeding, so dataloader shuffle/init drift is
        # at least reduced in principle, even if not perfectly eliminated
        torch.manual_seed(seed + trial.number)
        np.random.seed(seed + trial.number)

        optimiser_name = trial.suggest_categorical("optimiser", ["Adam", "AdamW"])
        lr             = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        weight_decay   = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

        backbone = build_backbone(model_name, pretrained=True).to(DEVICE)
        with torch.no_grad():
            _dummy = torch.zeros(2, 3, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
            feat_dim = backbone(_dummy).shape[1]
        del _dummy

        patch_model = PatchClassifier(backbone, feat_dim).to(DEVICE)

        pos_weight_val = class_weight_dict_[1] / class_weight_dict_[0]
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(pos_weight_val, device=DEVICE)
        )

        if optimiser_name == "AdamW":
            optimiser = optim.AdamW(patch_model.parameters(), lr=lr, weight_decay=weight_decay)
        else:
            optimiser = optim.Adam(patch_model.parameters(), lr=lr, weight_decay=weight_decay)

        n_workers = min(4, os.cpu_count() or 1)
        train_dl = DataLoader(PatchDataset(X_tr_, y_tr_), batch_size=BS_STAGE1,
                               shuffle=True, num_workers=n_workers, pin_memory=True,
                               persistent_workers=(n_workers > 0))
        val_dl   = DataLoader(PatchDataset(X_val_, y_val_), batch_size=BS_STAGE1,
                               shuffle=False, num_workers=n_workers, pin_memory=True,
                               persistent_workers=(n_workers > 0))

        scaler = torch.amp.GradScaler('cuda')
        best_val_loss, patience_ctr = float("inf"), 0

        for epoch in range(1, max_epochs + 1):
            patch_model.train()
            for patches, labels in train_dl:
                patches = patches.to(DEVICE, non_blocking=True)
                labels  = labels.to(DEVICE, non_blocking=True)
                patches = patches.repeat(1, 3, 1, 1)
                patches = (patches - _mean_gpu) / _std_gpu
                optimiser.zero_grad()
                with torch.amp.autocast('cuda'):
                    logits = patch_model(patches)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimiser)
                scaler.update()

            patch_model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for patches, labels in val_dl:
                    patches = patches.to(DEVICE, non_blocking=True)
                    labels  = labels.to(DEVICE, non_blocking=True)
                    patches = patches.repeat(1, 3, 1, 1)
                    patches = (patches - _mean_gpu) / _std_gpu
                    with torch.amp.autocast('cuda'):
                        logits = patch_model(patches)
                        loss_val = criterion(logits, labels)
                    val_loss += loss_val.item() * len(labels)
            val_loss /= len(val_dl.dataset)

            trial.report(val_loss, epoch)
            if trial.should_prune():
                del backbone, patch_model, optimiser
                gc.collect(); torch.cuda.empty_cache()
                raise optuna.TrialPruned()

            if val_loss < best_val_loss:
                best_val_loss, patience_ctr = val_loss, 0
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        del backbone, patch_model, optimiser
        gc.collect(); torch.cuda.empty_cache()
        return best_val_loss

    return objective

In [6]:
# Backbone configs + cache warmup
stage1_search_configs = [
    ("efficientnet_b0","effnet_b0", BASE_SEED + 0),
    ("convnext_nano","convnext_nano", BASE_SEED + 1),
    ("swin_tiny_patch4_window7_224", "swin_t",  BASE_SEED + 2),
]

print("Warming up backbone cache...")
for model_name, _, _ in stage1_search_configs:
    _ = timm.create_model(model_name, pretrained=True)
print("All backbone weights cached locally!\n")

Warming up backbone cache...


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/62.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

All backbone weights cached locally!



In [7]:
# Run searches
stage1_best_params = {}

for model_name, tag, seed in stage1_search_configs:
    print(f"\n{'='*60}\nStage 1 Optuna search: {tag} (seed={seed}, {N_TRIALS_STAGE1} trials)\n{'='*60}")

    sampler = TPESampler(seed=seed)   # CHANGED: per-architecture seed, not shared
    pruner  = MedianPruner(n_startup_trials=5, n_warmup_steps=2)   # CHANGED: more startup trials since budget is bigger
    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

    objective = make_stage1_objective(
        model_name, X_tr, y_tr, X_val, y_val, class_weight_dict,
        seed=seed, max_epochs=8, patience=3
    )

    def save_partial(study, trial, tag=tag, model_name=model_name):
        # CHANGED: tag/model_name passed as default args, not closed over 
        # removes any risk of stale-variable bleed between loop iterations
        partial = {
            "model_name": model_name,
            "best_params": study.best_params,
            "best_value": study.best_value,
            "n_trials_completed": len(study.trials),
            "n_trials_target": N_TRIALS_STAGE1,
        }
        with open(OUT / f"{tag}_stage1_partial.json", "w") as f:
            json.dump(partial, f, indent=2)

    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS_STAGE1, callbacks=[save_partial])
    elapsed = (time.time() - t0) / 60

    print(f"{tag} Stage 1 search complete — {elapsed:.1f} min")
    print("Best params:", study.best_params)
    print("Best val_loss:", study.best_value)

    stage1_best_params[tag] = {
        "model_name": model_name,
        "seed": seed,
        "best_params": study.best_params,
        "best_value": study.best_value,
        "n_trials": N_TRIALS_STAGE1,
        "search_time_min": elapsed,
    }

    with open(OUT / f"{tag}_stage1_optuna_study.json", "w") as f:
        json.dump(stage1_best_params[tag], f, indent=2)

print("\nAll Stage 1 searches complete:")
print(json.dumps(stage1_best_params, indent=2))

[I 2026-08-30 21:47:28,537] A new study created in memory with name: no-name-f43ac4c7-7521-4888-b1c6-a74a610808cd



Stage 1 Optuna search: effnet_b0 (seed=42, 20 trials)


[I 2026-08-30 21:56:40,502] Trial 0 finished with value: 0.6980614182993776 and parameters: {'optimiser': 'AdamW', 'lr': 0.001570297088405539, 'weight_decay': 0.0002481040974867811}. Best is trial 0 with value: 0.6980614182993776.
[I 2026-08-30 22:03:49,565] Trial 1 finished with value: 0.6683727527621319 and parameters: {'optimiser': 'Adam', 'lr': 1.493656855461762e-05, 'weight_decay': 0.0029154431891537554}. Best is trial 1 with value: 0.6683727527621319.
[I 2026-08-30 22:10:56,851] Trial 2 finished with value: 0.6606703738120254 and parameters: {'optimiser': 'AdamW', 'lr': 1.1527987128232396e-05, 'weight_decay': 0.00757947995334801}. Best is trial 2 with value: 0.6606703738120254.
[I 2026-08-30 22:18:05,888] Trial 3 finished with value: 0.6666581228524902 and parameters: {'optimiser': 'Adam', 'lr': 3.511356313970405e-05, 'weight_decay': 5.415244119402541e-06}. Best is trial 2 with value: 0.6606703738120254.
[I 2026-08-30 22:25:15,268] Trial 4 finished with value: 0.7037367441692677 

effnet_b0 Stage 1 search complete — 133.9 min
Best params: {'optimiser': 'AdamW', 'lr': 1.1527987128232396e-05, 'weight_decay': 0.00757947995334801}
Best val_loss: 0.6606703738120254

Stage 1 Optuna search: convnext_nano (seed=43, 20 trials)


[I 2026-08-31 00:10:49,432] Trial 0 finished with value: 0.6626376213540793 and parameters: {'optimiser': 'AdamW', 'lr': 2.512886612135209e-05, 'weight_decay': 9.169770785664835e-06}. Best is trial 0 with value: 0.6626376213540793.
[I 2026-08-31 00:19:24,373] Trial 1 finished with value: 0.6848346331365962 and parameters: {'optimiser': 'AdamW', 'lr': 0.0009960259174680093, 'weight_decay': 0.00014609954019847445}. Best is trial 0 with value: 0.6626376213540793.
[I 2026-08-31 00:36:44,850] Trial 2 finished with value: 0.6547080665449881 and parameters: {'optimiser': 'AdamW', 'lr': 0.0001530558928986471, 'weight_decay': 0.0016150593059496643}. Best is trial 2 with value: 0.6547080665449881.
[I 2026-08-31 00:45:01,767] Trial 3 finished with value: 0.6870653141652243 and parameters: {'optimiser': 'Adam', 'lr': 0.003980576022940335, 'weight_decay': 7.658010959494788e-06}. Best is trial 2 with value: 0.6547080665449881.
[I 2026-08-31 00:53:59,379] Trial 4 finished with value: 0.66575577755032

convnext_nano Stage 1 search complete — 169.6 min
Best params: {'optimiser': 'Adam', 'lr': 2.0514306954965613e-05, 'weight_decay': 5.292469708762985e-06}
Best val_loss: 0.6426443300390681

Stage 1 Optuna search: swin_t (seed=44, 20 trials)


[I 2026-08-31 03:10:54,765] Trial 0 finished with value: 0.6928633959751903 and parameters: {'optimiser': 'Adam', 'lr': 0.0017136473158328156, 'weight_decay': 2.766962957045446e-05}. Best is trial 0 with value: 0.6928633959751903.
[I 2026-08-31 03:40:37,770] Trial 1 finished with value: 0.6454434453843204 and parameters: {'optimiser': 'AdamW', 'lr': 0.00015182337912426777, 'weight_decay': 4.3280317638853594e-05}. Best is trial 1 with value: 0.6454434453843204.
[I 2026-08-31 04:11:41,511] Trial 2 finished with value: 0.6886505263942431 and parameters: {'optimiser': 'AdamW', 'lr': 0.007613400488234838, 'weight_decay': 6.706322521551027e-05}. Best is trial 1 with value: 0.6454434453843204.
[I 2026-08-31 04:37:18,748] Trial 3 finished with value: 0.6524677880222081 and parameters: {'optimiser': 'Adam', 'lr': 4.5050188036947596e-05, 'weight_decay': 0.00675909073123137}. Best is trial 1 with value: 0.6454434453843204.
[I 2026-08-31 04:57:33,331] Trial 4 finished with value: 0.687510974673376

swin_t Stage 1 search complete — 371.3 min
Best params: {'optimiser': 'Adam', 'lr': 9.114483750173519e-05, 'weight_decay': 0.00023846976987051193}
Best val_loss: 0.6385428700367021

All Stage 1 searches complete:
{
  "effnet_b0": {
    "model_name": "efficientnet_b0",
    "seed": 42,
    "best_params": {
      "optimiser": "AdamW",
      "lr": 1.1527987128232396e-05,
      "weight_decay": 0.00757947995334801
    },
    "best_value": 0.6606703738120254,
    "n_trials": 20,
    "search_time_min": 133.86305050849916
  },
  "convnext_nano": {
    "model_name": "convnext_nano",
    "seed": 43,
    "best_params": {
      "optimiser": "Adam",
      "lr": 2.0514306954965613e-05,
      "weight_decay": 5.292469708762985e-06
    },
    "best_value": 0.6426443300390681,
    "n_trials": 20,
    "search_time_min": 169.62129316329955
  },
  "swin_t": {
    "model_name": "swin_tiny_patch4_window7_224",
    "seed": 44,
    "best_params": {
      "optimiser": "Adam",
      "lr": 9.114483750173519e-05,
 